In [4]:
from pathlib import Path

import polars as pl

FILTERED_DIR = Path("../data/1_filtered")
TOKENIZED_DIR = Path("../data/2_tokenized")

## Stage 1: Filtered Games (UCI + evals + FEN)


In [5]:
# Load filtered games
filtered_files = sorted(FILTERED_DIR.glob("*.parquet"))[:3]
print("Filtered game files:", len(list(FILTERED_DIR.glob("*.parquet"))))
print("Sample files:", [f.name for f in filtered_files])

Filtered game files: 11
Sample files: ['filtered_2013-01.parquet', 'filtered_2013-02.parquet', 'filtered_2013-03.parquet']


In [6]:
# Load sample filtered game
filtered_sample = pl.read_parquet(filtered_files[0], n_rows=5)
print("Columns:", filtered_sample.columns)
print("\nSample row:")
filtered_sample

Columns: ['lichess_id', 'uci_moves', 'evals_cp', 'evals_raw', 'is_check', 'is_capture', 'piece_moved', 'promotion', 'is_en_passant', 'white_rating', 'black_rating', 'result', 'game_end_reason', 'time_initial', 'time_increment', 'utc_timestamp', 'opening', 'eco', 'ply_count', 'fen']

Sample row:


lichess_id,uci_moves,evals_cp,evals_raw,is_check,is_capture,piece_moved,promotion,is_en_passant,white_rating,black_rating,result,game_end_reason,time_initial,time_increment,utc_timestamp,opening,eco,ply_count,fen
str,str,list[i16],list[i16],list[bool],list[bool],list[str],list[str],list[bool],i16,i16,str,str,u16,u8,datetime[μs],str,str,u16,str
"""2irq4pg0""","""e2e4 e7e5 g1f3 d7d6 d2d4 e5d4 …","[12, 26, … null]","[12, 26, … 32767]","[false, false, … true]","[false, false, … true]","[""p"", ""p"", … ""q""]","["""", """", … """"]","[false, false, … false]",1785,1944,"""1-0""","""mate""",300,0,2013-01-01 06:15:59,"""Philidor Defense: Exchange Var…","""C41""",43,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""cbgpp8cc""","""e2e4 e7e5 f1c4 b8c6 c2c3 g7g6 …","[22, 20, … null]","[22, 20, … -32768]","[false, false, … true]","[false, false, … false]","[""p"", ""p"", … ""p""]","["""", """", … """"]","[false, false, … false]",1538,1607,"""0-1""","""mate""",600,10,2013-01-01 20:08:24,"""Bishop's Opening""","""C23""",52,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""8rpcvpav""","""e2e4 d7d5 e4d5 d8d5 b1c3 d5a5 …","[22, 41, … 0]","[22, 41, … 0]","[false, false, … true]","[false, false, … false]","[""p"", ""p"", … ""q""]","["""", """", … """"]","[false, false, … false]",1565,1598,"""1/2-1/2""","""agreement""",300,0,2013-01-02 02:45:21,"""Scandinavian Defense: Main Lin…","""B01""",89,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""282s2n5n""","""e2e4 e7e5 c2c4 g8f6 d2d3 c7c6 …","[17, 30, … null]","[17, 30, … -32763]","[false, false, … false]","[false, false, … false]","[""p"", ""p"", … ""p""]","["""", """", … """"]","[false, false, … false]",1835,1812,"""0-1""","""resignation""",540,0,2013-01-02 02:58:57,"""English Opening: The Whale""","""C20""",106,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"
"""etvda2dw""","""e2e4 e7e5 g1f3 b8c6 f1c4 g8f6 …","[27, 37, … -1533]","[27, 37, … -1533]","[false, false, … false]","[false, false, … true]","[""p"", ""p"", … ""q""]","["""", """", … """"]","[false, false, … false]",1692,1866,"""0-1""","""resignation""",300,0,2013-01-02 11:48:16,"""Italian Game: Two Knights Defe…","""C57""",28,"""rnbqkbnr/pppppppp/8/8/8/8/PPPP…"


## Stage 2: Tokenized (train/val ready)


In [33]:
# Load tokenized datasets
pretrain_path = TOKENIZED_DIR / "pretrain.parquet"
eval_path = TOKENIZED_DIR / "eval.parquet"

pretrain_sample = pl.read_parquet(pretrain_path, n_rows=1000)
eval_sample = pl.read_parquet(eval_path, n_rows=5)

print("Pretrain shape:", pl.read_parquet(pretrain_path).shape)
print("Eval shape:", pl.read_parquet(eval_path).shape)
print("\nPretrain columns:", pretrain_sample.columns)
print("\nSample tokenized row:")
pretrain_sample

Pretrain shape: (12947, 1)
Eval shape: (17, 1)

Pretrain columns: ['token_ids']

Sample tokenized row:


token_ids
list[u16]
"[0, 3, … 1]"
"[0, 3, … 1]"
"[0, 3, … 1]"
"[0, 4, … 1]"
"[0, 4, … 1]"
…
"[0, 4, … 1]"
"[0, 3, … 1]"
"[0, 4, … 1]"


## Token functions


In [34]:
from krasnal.tokens import (
    GAME_END_ID,
    GAME_START_ID,
    ID_TO_MOVE,
)

### Opening analysis


In [35]:
# Load filtered games and analyze openings
raw_lf = pl.scan_parquet(str(FILTERED_DIR / "*.parquet"))

openings = (
    raw_lf.select("opening")
    .collect()["opening"]
    .str.split(":")
    .list.get(0)
    .str.split(",")
    .list.get(0)
    .str.replace(r"#\d+", "")
    .str.strip_chars()
    .str.replace_all(r"\s+", " ")
    .unique()
    .sort()
)

print("unique normalized openings:", len(openings))
for opening in openings:
    print(opening)

unique normalized openings: 132
Alekhine Defense
Amar Opening
Anderssen Opening
Barnes Defense
Benko Gambit
Benko Gambit Accepted
Benko Gambit Declined
Benoni Defense
Bird Opening
Bishop's Opening
Blackmar-Diemer Gambit
Blackmar-Diemer Gambit Declined
Blumenfeld Countergambit
Bogo-Indian Defense
Borg Defense
Bronstein Gambit
Budapest Defense
Canard Opening
Caro-Kann Defense
Carr Defense
Catalan Opening
Center Game
Colle System
Creepy Crawly Formation
Czech Defense
Danish Gambit
Danish Gambit Accepted
Danish Gambit Declined
Duras Gambit
Dutch Defense
East Indian Defense
Elephant Gambit
English Defense
English Opening
Englund Gambit
Englund Gambit Complex
Englund Gambit Complex Declined
Englund Gambit Declined
Four Knights Game
Franco-Benoni Defense
French Defense
Gedult's Opening
Giuoco Piano
Goldsmith Defense
Grob Opening
Gruenfeld Defense
Guatemala Defense
Gunderam Defense
Horwitz Defense
Hungarian Opening
Indian Game
Italian Game
Kadas Opening
Kangaroo Defense
King's Gambit
King's Ga

### Sequence statistics


In [36]:
# length of the longest game in raw data (by move count)
raw_with_lengths = raw_lf.select(
    pl.col("uci_moves").str.split(" ").list.len().alias("move_count"),
    pl.col("uci_moves"),
).collect()

max_length = raw_with_lengths["move_count"].max()
max_idx = raw_with_lengths["move_count"].arg_max()
longest_game_moves = raw_with_lengths[max_idx, "uci_moves"]

print("longest game length (moves):", max_length)
print("example longest game:")
print(longest_game_moves)

longest game length (moves): 278
example longest game:
e2e4 g7g6 g1f3 f8g7 f1c4 d7d6 e1g1 g8f6 d2d3 e8g8 b1c3 b8d7 a2a3 f6g4 c3d5 c7c6 d5e3 d7e5 f3e5 g4e5 c4a2 g8h8 d3d4 e5d7 c2c3 d7f6 d1f3 e7e6 h2h3 d6d5 e4e5 f6e4 a2b1 e4g5 f3g3 f7f6 h3h4 g5f7 e3g4 f6e5 d4e5 d8e7 f1e1 a7a5 c1f4 b7b5 b1d3 c8b7 f4d2 c6c5 d3b5 c5c4 b5a4 e7c7 f2f4 a8b8 g3e3 b7c6 a4c6 c7c6 a1b1 f7h6 g4h6 g7h6 g2g4 h6f4 e3d4 f4d2 d4d2 c6c5 g1g2 f8f7 e1f1 b8f8 f1f7 f8f7 b1f1 f7f8 f1f8 c5f8 d2f2 f8d8 g4g5 h8g8 g2g3 d8c7 f2e2 g8g7 g3g4 c7b8 e2f2 b8e5 f2a7 g7g8 a7a8 g8g7 a8b7 g7g8 b7c8 g8f7 c8d7 f7g8 d7e8 g8g7 e8e7 g7g8 e7d8 g8g7 d8d7 g7g8 d7d8 g8g7 d8a5 e5e4 g4g3 e4e1 g3f3 e1e4 f3g3 e4e5 g3h3 e5f5 h3h2 f5f4 h2g2 f4g4 g2f1 g4h3 f1e2 h3g2 e2e3 g2e4 e3d2 e4g2 d2c1 g2f1 c1c2 f1f5 c2c1 f5f4 c1b1 f4h4 a5c7 g7f8 c7d8 f8f7 d8d7 f7f8 d7e6 h4g5 a3a4 g5g1 b1a2 g1c5 e6f6 f8g8 f6e6 g8g7 e6e5 g7h6 e5f4 g6g5 f4h2 h6g7 h2h5 h7h6 h5g4 g7f6 g4f3 f6g6 f3h3 c5c6 a2a3 h6h5 h3g2 h5h4 g2c2 g6h5 c2f5 c6d6 a3a2 d6c6 a2b1 c6a4 f5d5 h4h3 d5f3 h5h4 f3f2 

In [37]:
# count number of <GAME> and </GAME> tokens in pretrain parquet
flat_tokens = pl.col("token_ids").explode()
counts = pretrain_sample.select(
    flat_tokens.eq(GAME_START_ID).sum().alias("game_start_count"),
    flat_tokens.eq(GAME_END_ID).sum().alias("game_end_count"),
)

print("Tokenized column checked: token_ids")
print("<GAME> token ID:", GAME_START_ID)
print("</GAME> token ID:", GAME_END_ID)
print("total <GAME> tokens in sample:", int(counts["game_start_count"][0]))
print("total </GAME> tokens in sample:", int(counts["game_end_count"][0]))

Tokenized column checked: token_ids
<GAME> token ID: 0
</GAME> token ID: 1
total <GAME> tokens in sample: 1000
total </GAME> tokens in sample: 1000


### Tokenized game example


In [38]:
# Pick shortest game from pretrain_sample
# example_game = pretrain_sample.head(1).to_dicts()[0]
example_game = pretrain_sample.sort(pl.col("token_ids").list.len()).head(1).to_dicts()[0]
token_ids = example_game["token_ids"]

print("Token IDs:", token_ids)
print("Total tokens:", len(token_ids))

Token IDs: [0, 3, 11, 11, 1541, 1808, 20, 21, 1105, 3064, 16, 18, 1881, 3558, 2777, 16, 17, 1]
Total tokens: 18


In [39]:
# Same game decoded as string tokens
tokens_decoded = [ID_TO_MOVE.get(tid, f"<{tid}>") for tid in token_ids]

print("Tokens as strings:")
for i, token in enumerate(tokens_decoded):
    print(f"  {i}: {token}")

print(f"\nTotal tokens: {len(tokens_decoded)}")

Tokens as strings:
  0: <game_start>
  1: <white_won>
  2: <elo_1500_1999>
  3: <elo_1500_1999>
  4: w:e2e4
  5: b:e7e5
  6: <what_piece>
  7: <pawn>
  8: w:d1h5
  9: b:b8c6
  10: <is_check>
  11: <no_check>
  12: w:f1c4
  13: b:g8f6
  14: w:h5f7
  15: <is_check>
  16: <yes_check>
  17: <game_end>

Total tokens: 18
